# ResNet-18 (CIFAR-adapted) — train on Colab, compress locally

Runs on Colab's free GPU. Produces a `.pt` file at the end containing only the
model's `state_dict()` — deliberately *not* a pickled module, so it doesn't
depend on this notebook's classes being importable later. Download it and hand
it to `experiments/import_from_colab.py` in the compresskit repo, which wraps
it into a checkpoint the rest of the toolkit (Pipeline, CLI, demo notebook)
can load directly.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Architecture

Copied verbatim from `compresskit/models.py`'s `BasicBlock` / `ResNet18Cifar` —
same class names, same layer order, same parameter shapes. Do not edit this
cell; any structural change breaks the state_dict handoff.

In [ ]:
CIFAR10_CLASSES = 10


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        if stride != 1 or in_planes != planes:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        else:
            self.downsample = None

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + identity)


class ResNet18Cifar(nn.Module):
    def __init__(self, num_classes=CIFAR10_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.in_planes = 64
        self.layer1 = self._make_stage(64, 2, 1)
        self.layer2 = self._make_stage(128, 2, 2)
        self.layer3 = self._make_stage(256, 2, 2)
        self.layer4 = self._make_stage(512, 2, 2)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def _make_stage(self, planes, num_blocks, stride):
        layers = []
        for i in range(num_blocks):
            layers.append(BasicBlock(self.in_planes, planes, stride if i == 0 else 1))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x).flatten(1)
        return self.fc(x)


model = ResNet18Cifar().to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## Data — CIFAR-10, same normalisation stats as the local repo

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_ds = datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
test_ds = datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)

## Train

60 epochs, SGD + cosine LR + warmup — same recipe the local ResNet-20/56 baselines used.

In [ ]:
EPOCHS = 60
LR = 0.1
WARMUP_EPOCHS = 1


def cosine_lr(base_lr, epoch, total_epochs, warmup_epochs):
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / max(warmup_epochs, 1)
    progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
    return 0.5 * base_lr * (1 + torch.cos(torch.tensor(progress * 3.141592653589793))).item()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = seen = 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        correct += int((model(images).argmax(1) == targets).sum())
        seen += targets.size(0)
    return correct / seen


optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9,
                            weight_decay=5e-4, nesterov=True)

best_acc = 0.0
best_state = None

for epoch in range(EPOCHS):
    started = time.perf_counter()
    model.train()
    running_loss = correct = seen = 0
    batches = len(train_loader)

    for i, (images, targets) in enumerate(train_loader):
        for group in optimizer.param_groups:
            group["lr"] = cosine_lr(LR, epoch + i / batches, EPOCHS, WARMUP_EPOCHS)

        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = F.cross_entropy(logits, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * targets.size(0)
        correct += int((logits.argmax(1) == targets).sum())
        seen += targets.size(0)

    val_acc = evaluate(model, test_loader)
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"epoch {epoch+1:3d}/{EPOCHS}  loss {running_loss/seen:.3f}  "
          f"train {correct/seen*100:.2f}%  val {val_acc*100:.2f}%  "
          f"lr {optimizer.param_groups[0]['lr']:.4f}  {time.perf_counter()-started:.1f}s")

print(f"\nbest val accuracy: {best_acc*100:.2f}%")

## Save and download

Only the `state_dict` — no pickled classes, so this file has no dependency on
this notebook or on Colab's environment. `import_from_colab.py` in the local
repo turns this into a full compresskit checkpoint.

In [ ]:
torch.save({"state_dict": best_state, "arch": "resnet18", "accuracy": best_acc},
           "resnet18_cifar10.pt")
print("saved resnet18_cifar10.pt")

from google.colab import files
files.download("resnet18_cifar10.pt")